# 03. Speculative Decoding: Lossless Rejection Sampling and Drafting Paradigms

- **Category:** 15_compression_peft_and_speculative_inference / 02_adaptive_kv_speculative_and_merging
- **Lineage & Citations:** Leviathan et al. (ICML 2023, *Fast Inference from Transformers via Speculative Decoding*); Chen et al. (ICML 2023, *Accelerating Large Language Model Decoding with Speculative Sampling*); Saxena (2023, *Prompt Lookup Decoding*).
- **Purpose:** Implement lossless speculative rejection sampling, compute theoretical wall-clock speedup curves, and construct zero-parameter prompt lookup n-gram drafting engines.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [KV Cache Paged Radix H2O Snapkv and Yoco](02_kv_cache_paged_radix_h2o_snapkv_and_yoco.ipynb) $\to$ `🔗 Downstream Syntheses:` [Tree Speculative Medusa Eagle GoT and Jacobi](04_tree_speculative_medusa_eagle_got_and_jacobi.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use speculative decoding when operating at high server batch concurrency ($B \ge 64$); the GPU is already compute-bound and draft model execution creates contention that diminishes aggregate serving throughput. Do not use speculative decoding when target and draft models have mismatched tokenizers.

## Mathematical Foundations & Governing Equations

### 1. Acceptance Ratio & Decision Rule
For candidate token $x_i \sim q(x)$:
$$\alpha_i = \min\left(1, \frac{p(x_i)}{q(x_i)}\right), \quad \text{Accept} \iff u_i \le \alpha_i, \quad u_i \sim \mathcal{U}(0, 1)$$

### 2. Lossless Residual Distribution
If token $x_i$ is rejected, resample replacement token from normalized residual distribution:
$$p'(x) = \frac{\max(0, p(x) - q(x))}{\sum_y \max(0, p(y) - q(y))}, \quad P(\text{emit } x) \equiv p(x)$$

### 3. Expected Speedup & Lookahead Horizon $\gamma$
Given token acceptance rate $\beta = 1 - D_{\text{TV}}(p, q)$ and cost ratio $c = T_{\text{draft}} / T_{\text{target}}$:
$$\mathbb{E}[N] = \frac{1 - \beta^{\gamma+1}}{1 - \beta}, \quad \text{Speedup} = \frac{\mathbb{E}[N]}{\gamma c + 1}$$

### Complexity
Reduces sequential target HBM memory access passes from $N$ to $N / \mathbb{E}[N]$, yielding $2\times - 3.5\times$ end-to-end speedups.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def speculative_rejection_step(target_probs, draft_probs, draft_tokens, uniform_rands):
    batch, gamma, vocab = target_probs.shape
    accepted_mask = torch.zeros((batch, gamma), dtype=torch.float32, device=target_probs.device)
    num_accepted = torch.zeros((batch,), dtype=torch.float32, device=target_probs.device)
    residual_probs = torch.zeros((batch, vocab), dtype=torch.float32, device=target_probs.device)
    for b in range(batch):
        rejected_pos = -1
        for i in range(gamma):
            t_id = int(draft_tokens[b, i].item())
            p_val, q_val = target_probs[b, i, t_id].item(), draft_probs[b, i, t_id].item()
            ratio = min(1.0, p_val / max(q_val, 1e-8))
            if uniform_rands[b, i].item() <= ratio:
                accepted_mask[b, i] = 1.0
                num_accepted[b] += 1.0
            else:
                rejected_pos = i
                break
        if rejected_pos >= 0:
            diff = torch.clamp(target_probs[b, rejected_pos] - draft_probs[b, rejected_pos], min=0.0)
            norm = torch.sum(diff)
            residual_probs[b] = diff / norm if norm > 1e-8 else target_probs[b, rejected_pos]
        else:
            residual_probs[b] = target_probs[b, -1]
    return accepted_mask, num_accepted, residual_probs

def speculative_speedup_theoretical(gamma, beta, cost_ratio):
    exp_tokens = float(gamma + 1) if abs(beta - 1.0) < 1e-6 else (1.0 - beta ** (gamma + 1)) / (1.0 - beta)
    return exp_tokens / (gamma * cost_ratio + 1.0)

def prompt_lookup_ngram_draft(input_ids, ngram_size=3, gamma=4):
    seq_len = input_ids.shape[0]
    if seq_len < ngram_size + 1:
        return torch.zeros(gamma, dtype=torch.float32, device=input_ids.device)
    suffix = input_ids[-ngram_size:]
    best_pos = -1
    for i in range(seq_len - ngram_size - 1, -1, -1):
        if torch.equal(input_ids[i : i + ngram_size], suffix):
            best_pos = i + ngram_size
            break
    draft = [input_ids[best_pos + k].item() if best_pos >= 0 and best_pos + k < seq_len else 0.0 for k in range(gamma)]
    return torch.tensor(draft, dtype=torch.float32, device=input_ids.device)

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

acc_mask, n_acc, res_p = speculative_rejection_step(weights["target_probs"], weights["draft_probs"], weights["draft_tokens"], weights["uniform_rands"])
assert float(torch.max(torch.abs(acc_mask - weights["accepted_mask"]))) < 1e-5
assert float(torch.max(torch.abs(n_acc - weights["num_accepted"]))) < 1e-5
assert float(torch.max(torch.abs(res_p - weights["residual_probs"]))) < 1e-5

draft = prompt_lookup_ngram_draft(weights["prompt_seq"], 3, 4)
assert float(torch.max(torch.abs(draft - weights["draft_ngram"]))) < 1e-5
print("All speculative decoding operators achieve verified golden parity (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_spec = benchmark_fn(lambda: speculative_rejection_step(weights["target_probs"], weights["draft_probs"], weights["draft_tokens"], weights["uniform_rands"]), iters=100, name="Speculative Verify Step")
b_prompt = benchmark_fn(lambda: prompt_lookup_ngram_draft(weights["prompt_seq"], 3, 4), iters=100, name="Prompt Lookup Drafting")
print(format_benchmark_table([b_spec, b_prompt]))

for gamma in [3, 5, 7]:
    s = speculative_speedup_theoretical(gamma=gamma, beta=0.8, cost_ratio=0.05)
    print(f"Theoretical Speedup (gamma={gamma}, beta=0.8): {s:.2f}x")

In [ ]:
# Pathological Misuse: Naive Resampling upon Rejection corrupts output probability distribution
p = torch.tensor([0.7, 0.2, 0.1])  # Target
q = torch.tensor([0.1, 0.8, 0.1])  # Draft (severely biased toward token 1)

# 1. Flawed naive fallback: upon rejection, sampling directly from p(x) without subtracting q(x)
# 2. Correct residual distribution: p'(x) = max(0, p - q) normalized
diff = torch.clamp(p - q, min=0.0)
p_prime = diff / torch.sum(diff)

# Notice p_prime has 0 probability on token 1 because q already over-sampled it!
assert p_prime[1].item() == 0.0, "Residual distribution must zero out tokens over-covered by draft!"
assert p_prime[0].item() > p[0].item(), "Residual distribution must boost tokens under-covered by draft!"
print(f"Correct Residual Distribution: {p_prime.tolist()} restores exact target expectation!")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Provably Lossless Acceleration**: Speculative rejection sampling guarantees identical sample distributions to the target foundation model while delivering $2\times-3.5\times$ speedups.
2. **The Residual Correction**: Sampling from normalized $\max(0, p - q)$ upon rejection cancels out draft model inductive biases exactly.
3. **Zero-Parameter Drafting**: Prompt lookup n-gram search provides instant speculative candidates for structured, repetitive, or retrieval-augmented text without loading extra models.

---
### Cross-Domain Generalisation Matrix
- **Computer Graphics:** Speculative rejection sampling directly mirrors shadow ray and path-tracing importance sampling.
- **Database Query Engines:** Speculative execution in transactional databases executes low-confidence query paths concurrently.
- **Compiler Optimization:** Branch prediction and speculative execution in out-of-order superscalar CPUs.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.md](../../../markdown/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.md)
- **PyTorch 2.x Module:** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.py](../../../pytorch2/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.py)
- **Functional JAX Module:** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.py](../../../jax/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.py)
- **Native Rust Candle Track:** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.rs](../../../candle/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.rs)
- **Hardware Kernel (CPU / WGSL):** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.rs](../../../kernels/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.rs)
- **Golden SafeTensors:** [03_speculative_decoding_lossless_rejection_sampling_and_drafting.safetensors](../../../weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/03_speculative_decoding_lossless_rejection_sampling_and_drafting.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. KV Cache Paged Radix H2O Snapkv and Yoco](02_kv_cache_paged_radix_h2o_snapkv_and_yoco.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 04. Tree Speculative Medusa Eagle GoT and Jacobi](04_tree_speculative_medusa_eagle_got_and_jacobi.ipynb) &rarr;